In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from blox2.utils import make_scaled_trajectory

plt.rcParams.update({
    "font.size": 16,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
})

n_obs_0 = 10
# results_root, all_path, n_seeds, n_samples = "results/dft_multi", "../data/zinc_dft/properties.csv", 5, 2000
results_root, all_path, n_seeds, n_samples = "results/material_multi", "../data/material/properties.csv", 20, 400

# Sigma

dir_names = [
    "02-14_064844_random",
    "02-13_161719_lgb",
    "02-14_014816_lgb_batch100",
    "batch_penalty/02-17_184917_mat_lgb_batch100_dist_w0.5",
    "batch_penalty/02-17_185242_mat_lgb_batch100_stein_w0.5",
    "batch_stein_sigma/02-17_205744_mat_lgb_batch100_stein_w0.5_ps0.5",
    "batch_stein_sigma/02-17_212428_mat_lgb_batch100_stein_w0.5_ps5",
    "batch_stein_sigma/02-18_111254_mat_lgb_batch100_stein_w0.5_psauto",
    "batch_stein_sigma/02-17_212748_mat_lgb_batch100_stein_w0.5_ps50",
]

legends = None
legends = [
    "Random",
    "No batch",
    "No penalty",
    "Dist",
    "Stein (σ=1.0)",
    "Stein (σ=0.5)",
    "Stein (σ=5.0)",
    "Stein (σ=13.75, auto)",
    "Stein (σ=50.0)",
]

n_batches = [
    1,
    1,
    100,
    100,
    100,
    100,
    100,
    100,
    100,
]

def load_trajectories_for_dn(dn: str, all_path: str | None, n_seeds: int, results_root: str = "results") -> list[np.ndarray]:
    """
    Load scaled trajectories for a single experiment directory (dn) across multiple seeds.

    Args:
        dn: Experiment directory name under results_root.
        init_path: Path to initial observed properties CSV.
        all_path: Path to global properties CSV used for scaling. If None, no scaling.
        n_seeds: Number of seed_init_{i} folders to load (0, ..., n_seeds-1).
        results_root: Root directory containing experiment outputs.
    """
    trajectories = []
    for seed in range(n_seeds):
        init_path = Path(results_root) / dn / f"seed_init_{seed}" / "initial_observed_properties.csv"
        obs_path = Path(results_root) / dn / f"seed_init_{seed}" / "observation_history.csv"
        if not obs_path.exists():
            raise FileNotFoundError(f"Missing: {obs_path}")

        t = make_scaled_trajectory(init_path, str(obs_path), all_path)
        trajectories.append(t)
    return trajectories


def mean_and_se(arr: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = arr.mean(axis=0)
    se = arr.std(axis=0, ddof=1) / np.sqrt(arr.shape[0]) if arr.shape[0] >= 2 else np.zeros_like(mean)
    return mean, se

def plot_metric_across_seeds(trajs_by_dn: list[list[np.ndarray]], labels: list[str], metric_fn, n_obs_0: int, n_samples: int=None, batch_sizes: list[int]=None, ax=None, ci: bool=False, ci_scale: float=1.0):
    if ax is None:
        ax = plt.gca()

    if batch_sizes is None:
        batch_sizes = [1] * len(trajs_by_dn)

    for runs, label, b in zip(trajs_by_dn, labels, batch_sizes):
        metrics = []
        for t in runs:
            m = metric_fn(t)
            metrics.append(m)

        # Ensure same length across seeds (truncate to min length if needed)
        min_len = min(len(m) for m in metrics)
        if n_samples is not None:
            min_len = min(min_len, n_obs_0 + n_samples + 1)

        metrics = np.stack([m[:min_len] for m in metrics], axis=0)  # (n_seeds, T)
        mean, se = mean_and_se(metrics)

        b = int(b)
        if b <= 0:
            raise ValueError(f"batch size must be positive, got {b}")

        # use only multiples of batch size after n_obs_0
        idx = np.arange(n_obs_0, min_len, b)
        x = idx - n_obs_0
        y = mean[idx]

        ax.plot(x, y, label=label)

        if ci and metrics.shape[0] >= 2:
            band = ci_scale * se[idx]
            ax.fill_between(x, y - band, y + band, alpha=0.2)

    ax.legend()
    return ax

trajs_by_dn = []
for dn in dir_names:
    runs = load_trajectories_for_dn(dn=dn, all_path=all_path, n_seeds=n_seeds, results_root=results_root)
    trajs_by_dn.append(runs)
    
plt.figure()
for dn_runs, label in zip(trajs_by_dn, legends):
    for t in dn_runs:
        plt.scatter(t[:, 0], t[:, 1], alpha=0.2, s=0.2)
        
for label in legends: # legend for groups
    plt.scatter([], [], s=20, label=label)
plt.legend(markerscale=5)
plt.xlabel("Objective 1 (scaled)")
plt.ylabel("Objective 2 (scaled)")
plt.tight_layout()

In [ ]:
# union of buffers

from blox2 import buffer_union_area_trajectory

radius = 0.1
step = 2

def union_metric(t):
    return buffer_union_area_trajectory(t, radius=radius, step=step)

ax = plot_metric_across_seeds(trajs_by_dn=trajs_by_dn, labels=legends, metric_fn=union_metric, n_obs_0=n_obs_0, n_samples=n_samples, batch_sizes=n_batches, ci=True, ci_scale=1.0)
ax.set_xlabel("Number of samplings")
ax.set_ylabel(f"Area of buffers (ε={radius})")
plt.tight_layout()